# Text-to-SQL: Natural Language Queries for SF Parcels

This notebook demonstrates how to build a natural language → SQL pipeline using:
- **DataSF API** to fetch the parcels dataset into memory
- **DuckDB** as an in-memory SQL engine (fast, zero config, great pandas integration)
- **Claude API** (claude-haiku-4-5) to translate plain English into SQL
- **Prompt caching** so the schema isn't re-processed on every call

Dataset: [SF Parcels (Active and Retired)](https://data.sfgov.org/Geographic-Locations-and-Boundaries/Parcels-Active-and-Retired/acdm-wktn/about_data)  

> **Note:** This is a learning tool, not production code.

## 1. Setup

```bash
pip install anthropic requests pandas duckdb
```

```bash
export ANTHROPIC_API_KEY=sk-ant-...
```

In [20]:
import anthropic
import duckdb
import requests
import pandas as pd
import re
from dotenv import load_dotenv

load_dotenv()  # loads ANTHROPIC_API_KEY from .env if present

DATASET_ID   = "acdm-wktn"
DATA_URL     = f"https://data.sfgov.org/resource/{DATASET_ID}.json"
METADATA_URL = f"https://data.sfgov.org/api/views/{DATASET_ID}.json"
TABLE_NAME   = "parcels"
MODEL        = "claude-haiku-4-5-20251001"
FETCH_LIMIT  = 50_000   # rows to pull from DataSF (max per request: 50k)

client = anthropic.Anthropic()
print("Setup complete.")

Setup complete.


## 2. Fetch the Dataset

We pull from the DataSF API once and store everything in a pandas DataFrame. The Socrata API returns JSON arrays — each row is a dict of `{column: value}`.

Because Socrata encodes everything as strings in JSON, we run a type-coercion pass after loading: any column where every value parses as a number gets converted. DuckDB will then see those columns as `DOUBLE` instead of `VARCHAR`, which means Claude can write `WHERE yrbuilt > 2009` without casting.

In [21]:
def fetch_data(limit=FETCH_LIMIT):
    """Fetch rows from DataSF API. Socrata max per request is 50,000."""
    print(f"Fetching up to {limit:,} rows from DataSF...")
    resp = requests.get(DATA_URL, params={"$limit": str(limit)}, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    print(f"Got {len(data):,} rows.")
    return data


def coerce_numeric_columns(df):
    """Convert columns to numeric where every non-null value parses as a number."""
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass
    return df


raw_data = fetch_data()
df = pd.DataFrame(raw_data)
df = coerce_numeric_columns(df)

print(f"\nDataFrame shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumn types after coercion:")
print(df.dtypes)

Fetching up to 50,000 rows from DataSF...
Got 50,000 rows.

DataFrame shape: 50,000 rows × 37 columns

Column types after coercion:
mapblklot                    object
blklot                       object
block_num                    object
lot_num                      object
from_address_num            float64
to_address_num              float64
street_name                  object
street_type                  object
odd_even                     object
in_asr_secured_roll            bool
pw_recorded_map                bool
zoning_code                  object
zoning_district              object
date_map_add                 object
project_id_add               object
active                         bool
shape                        object
centroid_latitude           float64
centroid_longitude          float64
supdist                      object
supervisor_district         float64
supdistpad                  float64
numbertext                   object
supname                      object
anal

## 3. Load into DuckDB

DuckDB can register a pandas DataFrame directly as a virtual table — no data is copied, just a reference. From here, any DuckDB SQL query against `parcels` reads from the DataFrame in memory.

In [22]:
con = duckdb.connect()          # in-memory database, no file needed
con.register(TABLE_NAME, df)    # register the DataFrame as a table

# Verify it works
count = con.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}").fetchone()[0]
print(f"Table '{TABLE_NAME}' registered with {count:,} rows.")

# Peek at a few rows
con.execute(f"SELECT * FROM {TABLE_NAME} LIMIT 3").df()

Table 'parcels' registered with 50,000 rows.


,mapblklot,blklot,block_num,lot_num,from_address_num,to_address_num,street_name,street_type,odd_even,in_asr_secured_roll,...,planning_district,planning_district_number,data_as_of,data_loaded_at,date_rec_add,date_rec_drop,date_map_drop,project_id_drop,date_map_alt,project_id_alt
0,3584032,3584032,3584,032,3976.0,3976.0,19TH,ST,E,True,...,Central,7.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,None,None,None,None,None,None
1,3602085,3602085,3602,085,4043.0,4043.0,19TH,ST,O,True,...,Central,7.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,None,None,None,None,None,None
2,0692030,0692258,0692,258,NaN,NaN,None,None,None,False,...,Downtown,4.0,2026-04-23T03:57:00.000,2026-04-23T14:26:04.773,2025-05-07T00:00:00.000,None,None,None,None,None


## 4. Build the Schema for the System Prompt

We get column names and types from DuckDB's `DESCRIBE` command (so they reflect the actual in-memory types after coercion), then enrich them with human-readable names and descriptions from the Socrata metadata API.

This combined schema is what we'll embed in Claude's system prompt — and cache.

In [23]:
def get_schema_text(con, table_name, metadata_url):
    """Build a schema description string for the system prompt."""
    # Actual types from DuckDB
    schema_df = con.execute(f"DESCRIBE {table_name}").df()
    type_map = dict(zip(schema_df["column_name"], schema_df["column_type"]))

    # Human-readable names + descriptions from Socrata metadata
    resp = requests.get(metadata_url, timeout=30)
    resp.raise_for_status()
    socrata_cols = {
        col["fieldName"]: col
        for col in resp.json().get("columns", [])
        if not col["fieldName"].startswith(":")
    }

    lines = []
    for col_name, col_type in type_map.items():
        meta = socrata_cols.get(col_name, {})
        display_name = meta.get("name", col_name)
        description = meta.get("description", "").strip()
        line = f"- {col_name} ({col_type}): {display_name}"
        if description:
            line += f" — {description}"
        lines.append(line)
    return "\n".join(lines)


schema_text = get_schema_text(con, TABLE_NAME, METADATA_URL)

print(f"Schema: {schema_text.count(chr(10)) + 1} columns\n")
print("First 10 columns:")
print("\n".join(schema_text.split("\n")[:10]))

Schema: 37 columns

First 10 columns:
- mapblklot (VARCHAR): mapblklot — For parcels that are condominium lots and share the same 2D space, this is the lowest value  Assessor Parcel Number in the set.
- blklot (VARCHAR): blklot — Unique Assessor Parcel Number, combination of the Block Number and the Lot Number
- block_num (VARCHAR): block_num — Assessor Block Number
- lot_num (VARCHAR): lot_num — Assessor Lot Number
- from_address_num (DOUBLE): from_address_num — From Address Number, i.e., the lowest address house number
- to_address_num (DOUBLE): to_address_num — To Address Number, i.e., the highest address house number
- street_name (VARCHAR): street_name — Street Name
- street_type (VARCHAR): street_type — Street Type, or suffix (e.g., ST, BLVD, etc.)
- odd_even (VARCHAR): odd_even — Odd or Even Address Number Range
- in_asr_secured_roll (BOOLEAN): in_asr_secured_roll — Indicates the record is in the current Assessor Secured Roll


## 5. Build the System Prompt

The system prompt tells Claude:
1. It's writing DuckDB SQL against a table called `parcels`
2. What every column is named, typed, and means
3. To return only the SQL — no explanation, no markdown

We'll attach `cache_control` to this prompt so the schema is cached after the first call.

In [24]:
SYSTEM_PROMPT = f"""You are an expert at writing DuckDB SQL queries.

The database contains a single table called `{TABLE_NAME}` with the following columns:

{schema_text}

Rules:
- Return ONLY the raw SQL query. No explanation before or after. No alternatives. No commentary.
- Your entire response must be valid SQL that can be executed directly.
- Always include a LIMIT clause (default 100 unless the user asks for more or an aggregate).
- Use standard DuckDB SQL syntax (CTEs, window functions, and aggregates are all supported).
- String comparisons are case-sensitive; use LOWER() or UPPER() for case-insensitive matching.
- If you are unsure about a column name, use the closest match from the schema above.
"""

print(f"System prompt: {len(SYSTEM_PROMPT):,} characters (~{len(SYSTEM_PROMPT) // 4:,} tokens)")

System prompt: 4,542 characters (~1,135 tokens)


## 6. Natural Language → SQL

Claude returns a plain SQL string. We strip any markdown code fences in case they sneak in, then hand the query straight to DuckDB.

In [25]:
def nl_to_sql(question, verbose=True):
    """Translate a natural language question into a DuckDB SQL query."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": question}],
    )

    if verbose:
        u = response.usage
        print("Token usage:")
        print(f"  Regular input:  {u.input_tokens:>6,}  (full cost)")
        print(f"  Cache write:    {u.cache_creation_input_tokens:>6,}  (1.25x — first call only)")
        print(f"  Cache read:     {u.cache_read_input_tokens:>6,}  (0.1x — subsequent calls)")
        print(f"  Output:         {u.output_tokens:>6,}")

    raw = response.content[0].text.strip()

    # Strip markdown code fences
    raw = re.sub(r"^```(?:sql)?\s*", "", raw, flags=re.MULTILINE)
    raw = re.sub(r"\s*```$", "", raw, flags=re.MULTILINE).strip()

    # If the model added prose before/after the SQL, extract just the SQL block.
    # Look for the first SELECT, WITH, or other DML keyword and take from there.
    sql_match = re.search(r"((?:WITH|SELECT|INSERT|UPDATE|DELETE)\b.*)", raw, re.DOTALL | re.IGNORECASE)
    if sql_match:
        raw = sql_match.group(1).strip()

    # Drop anything after the first semicolon (trailing commentary)
    if ";" in raw:
        raw = raw[:raw.index(";") + 1].strip()

    return raw


print("nl_to_sql() defined.")

nl_to_sql() defined.


## 7. Execute + Display

In [26]:
def ask(question, verbose=True):
    """Natural language → SQL → DataFrame."""
    print(f"Question: {question}\n")

    sql = nl_to_sql(question, verbose=verbose)
    print(f"\nGenerated SQL:\n{sql}\n")

    result = con.execute(sql).df()
    print(f"Returned {len(result):,} rows, {len(result.columns)} columns")
    return result


print("ask() defined. Ready to query!")

ask() defined. Ready to query!


---
## 8. Example Queries

In [27]:
# Example 1 — basic filter
df1 = ask("find me all parcels built after 2009")

Question: find me all parcels built after 2009

Token usage:
  Regular input:   1,202  (full cost)
  Cache write:         0  (1.25x — first call only)
  Cache read:          0  (0.1x — subsequent calls)
  Output:             36

Generated SQL:
SELECT *
FROM parcels
WHERE date_map_add > '2009-12-31'
LIMIT 100;

Returned 100 rows, 37 columns


In [28]:
df1.head(10)

,mapblklot,blklot,block_num,lot_num,from_address_num,to_address_num,street_name,street_type,odd_even,in_asr_secured_roll,...,planning_district,planning_district_number,data_as_of,data_loaded_at,date_rec_add,date_rec_drop,date_map_drop,project_id_drop,date_map_alt,project_id_alt
0,0692030,0692258,0692,258,NaN,NaN,None,None,None,False,...,Downtown,4.0,2026-04-23T03:57:00.000,2026-04-23T14:26:04.773,2025-05-07T00:00:00.000,None,None,None,None,None
1,3603117,3603118,3603,118,3866.0,3866.0,21ST,ST,E,True,...,Central,7.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,2014-10-02T00:00:00.000,None,None,None,None,None
2,4041011,4041121,4041,121,NaN,NaN,None,None,None,False,...,South of Market,9.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,2016-11-22T00:00:00.000,None,None,None,None,None
3,0793104,0793131,0793,131,400.0,400.0,GROVE,ST,E,True,...,Western Addition,5.0,2026-04-23T03:54:00.000,2026-04-23T14:26:04.773,2015-01-07T00:00:00.000,None,None,None,None,None
4,1084012,1084020,1084,020,1.0,1.0,STANYAN,BLVD,O,True,...,Richmond,1.0,2026-04-23T03:54:00.000,2026-04-23T14:26:04.773,2018-05-10T00:00:00.000,None,None,None,None,None
5,0794059,0794130,0794,130,555.0,555.0,FULTON,ST,O,True,...,Western Addition,5.0,2026-04-23T03:54:00.000,2026-04-23T14:26:04.773,2018-09-17T00:00:00.000,None,None,None,None,None
6,3633077,3633078,3633,078,78.0,78.0,ALVARADO,ST,E,True,...,Mission,8.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,2015-07-08T00:00:00.000,None,None,None,None,None
7,4991743,4991887,4991,887,NaN,NaN,None,None,None,False,...,South Bayshore,10.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,2011-08-03T00:00:00.000,None,None,None,None,None
8,3521298,3521435,3521,435,NaN,NaN,None,None,None,False,...,Mission,8.0,2026-04-23T03:55:00.000,2026-04-23T14:26:04.773,2021-03-17T00:00:00.000,None,None,None,None,None
9,0692030,0692293,0692,293,NaN,NaN,None,None,None,False,...,Downtown,4.0,2026-04-23T03:57:00.000,2026-04-23T14:26:04.773,2025-05-07T00:00:00.000,None,None,None,None,None


In [29]:
# Example 2 — aggregation
df2 = ask("how many parcels are there for each land use type, sorted by count descending?")

Question: how many parcels are there for each land use type, sorted by count descending?

Token usage:
  Regular input:   1,210  (full cost)
  Cache write:         0  (1.25x — first call only)
  Cache read:          0  (0.1x — subsequent calls)
  Output:             52

Generated SQL:
SELECT 
  zoning_code,
  COUNT(*) as parcel_count
FROM parcels
GROUP BY zoning_code
ORDER BY parcel_count DESC
LIMIT 100;

Returned 100 rows, 2 columns


In [30]:
df2

,zoning_code,parcel_count
0,RH-1,11898
1,RH-2,8980
2,RH-3,3607
3,RTO-C,2819
4,RH-1(D),2480
...,...,...
95,CRNC,3
96,TI-OS|TI-R,3
97,PDR-1-D,3
98,PM-MU2|PM-OS|PM-R|PM-R|RM-1,3


In [ ]:
# Example 3 — something harder that would have been awkward in SoQL
df3 = ask("what is the average lot area per land use type, only for types with more than 100 parcels?")

In [ ]:
df3

In [ ]:
# Example 4 — window function (would not be possible in SoQL at all)
df4 = ask("for each land use type, show the 3 largest parcels by lot area")

In [ ]:
df4.head(20)

---
## 9. Prompt Caching: First Call vs. Subsequent Calls

The schema is large (~1K tokens). Without caching, every call pays the full input cost to process it. With `cache_control: {"type": "ephemeral"}`, the first call writes the cache (1.25x cost) and subsequent calls read from it (0.1x — a 92% discount).

The cache TTL is **5 minutes**. Each hit resets the timer.

In [31]:
def cost_of(usage):
    """Estimate cost in USD using Haiku 4.5 pricing."""
    return (
        usage.input_tokens                    *  1.00 / 1_000_000
        + usage.cache_creation_input_tokens   *  1.25 / 1_000_000
        + usage.cache_read_input_tokens       *  0.10 / 1_000_000
        + usage.output_tokens                 *  5.00 / 1_000_000
    )


def timed_call(question):
    """Call nl_to_sql and return (sql, usage, cost)."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=[{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": question}],
    )
    sql = re.sub(r"^```(?:sql)?\s*", "", resp.content[0].text.strip(), flags=re.MULTILINE)
    sql = re.sub(r"\s*```$", "", sql, flags=re.MULTILINE).strip()
    return sql, resp.usage, cost_of(resp.usage)


q = "show the 5 most common zip codes by parcel count"

_, u1, c1 = timed_call(q)
print("Call 1 (cache WRITE):")
print(f"  Regular input:  {u1.input_tokens:>6,}")
print(f"  Cache write:    {u1.cache_creation_input_tokens:>6,}  ← schema processed & stored")
print(f"  Cache read:     {u1.cache_read_input_tokens:>6,}")
print(f"  Output:         {u1.output_tokens:>6,}")
print(f"  Cost:           ${c1:.5f}\n")

_, u2, c2 = timed_call(q)
print("Call 2 (cache READ):")
print(f"  Regular input:  {u2.input_tokens:>6,}")
print(f"  Cache write:    {u2.cache_creation_input_tokens:>6,}")
print(f"  Cache read:     {u2.cache_read_input_tokens:>6,}  ← schema loaded from cache")
print(f"  Output:         {u2.output_tokens:>6,}")
print(f"  Cost:           ${c2:.5f}\n")

if c1 > 0:
    print(f"Call 2 is {(1 - c2/c1)*100:.1f}% cheaper than call 1")

Call 1 (cache WRITE):
  Regular input:   1,205
  Cache write:         0  ← schema processed & stored
  Cache read:          0
  Output:             79
  Cost:           $0.00160

Call 2 (cache READ):
  Regular input:   1,205
  Cache write:         0
  Cache read:          0  ← schema loaded from cache
  Output:            145
  Cost:           $0.00193

Call 2 is -20.6% cheaper than call 1


---
## 10. Try Your Own Query

In [ ]:
your_question = "show me the 10 oldest parcels still active"

df = ask(your_question)
df.head(10)

---
## Key Takeaways

1. **Fetch once, query many times** — pulling the dataset into DuckDB upfront means every subsequent query is a local in-memory operation. No network round-trip per question.

2. **DuckDB SQL vs SoQL** — DuckDB supports the full SQL surface: CTEs, window functions, `HAVING`, subqueries, arbitrary `JOIN`s. SoQL is URL-parameter-based and much more limited.

3. **Type coercion matters** — Socrata returns everything as strings in JSON. Converting numeric columns before loading lets Claude write `WHERE yrbuilt > 2009` naturally instead of needing `CAST`s everywhere.

4. **Schema-in-prompt** — Claude needs to know the exact column names and types. `DESCRIBE` gives the ground truth after coercion; the Socrata metadata adds human-readable context.

5. **Prompt caching** — The schema is stable for the lifetime of the notebook session. Caching it means you only pay 1.25x on the first call; everything after is 0.1x for that prefix.

6. **Model choice** — Haiku 4.5 handles straightforward SQL generation well at $1/$5 per 1M tokens. For complex multi-step queries or ambiguous phrasing, bump to `claude-sonnet-4-6` or `claude-opus-4-7` — one line change.

### What this isn't
No SQL injection defense (don't expose `ask()` to untrusted users), no pagination, no schema refresh when the dataset updates. Learning prototype only.